# Public-source note

This notebook preserves the original 23-state ATAC ChromHMM reordering logic. Outputs were stripped and private paths sanitized.

In [ ]:
import os, shlex, subprocess, sys
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
learnmodel_out_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_atac/numstates_23/"
reordered_out_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_atac/numstates_23_reordered_V2/"

emissions_in = Path(learnmodel_out_dir) / "emissions_23.txt"
emissions_out = Path(reordered_out_dir) / "emissions_23.txt"
emissions_out_with_Percent = Path(reordered_out_dir) / "emissions_23_with_Genome_percentage.txt"

overlap_in = Path(learnmodel_out_dir) / "ATAC_23_overlap.txt"
overlap_out = Path(reordered_out_dir) / "ATAC_23_overlap.txt"

seg_in = Path(learnmodel_out_dir) / "ATAC_23_segments.bed"
seg_out= Path(reordered_out_dir) / "ATAC_23_segments.bed"

emissions_out_pdf= Path(reordered_out_dir) / "ATAC_STACKED_ChromHMM_23_reordered.pdf"

In [ ]:
col_order=[
'MEF', 'MEF_d2_OSKM','MEF_d2_OSKM_mCEBPA', 'MEF_d2_OSKM_mCEBPB',  'MEF_d2_OSKM_hCEBPD','MEF_d2_mCEBPA', 'MEF_d2_mCEBPB', 
'NSC', 'NSC_d2_OSKM', 'NSC_d2_OSKM_mCEBPA', 'NSC_d2_OSKM_mCEBPB', 'NSC_d2_mCEBPA', 'NSC_d2_mCEBPB',
'ESC', 'ESC_d1_hCEBPA', 'ESC_d2_hCEBPA',   
'Bcell_Graf', 'Bcell_Graf_Cebpa18h', 'Bcell_Graf_OKSMd1', 'Bcell_Graf_OKSMd2'  ]

In [ ]:
desired_order = [6, 5,18,
                 22,7, 4,
                 23, 20, 21,
                 17,15, 16,
                 11, 9, 10,8,
                 1,3, 2, 19, 12,13, 14]

state_map = {old: new for new, old in enumerate(desired_order, start=1)}
state_map

In [ ]:
desired_order = [6, 5,18,
                 22,7, 4,
                 23, 20, 21,
                 17,15, 16,
                 11, 9, 10,8,
                 1,3, 2, 19, 12,13, 14,"Base"]

state_map_str = {str(old): new for new, old in enumerate(desired_order, start=1)}
state_map_str

In [ ]:
len(desired_order)

In [ ]:
# ---- create directory if it does not exist ----
reordered_path = Path(reordered_out_dir)

reordered_path.mkdir(parents=True, exist_ok=True)

print(f"Directory ready: {reordered_path}")

In [ ]:
# emissions_df=pd.read_csv(emissions_in, sep="\t")

# state_col = "State (Emission order)"
# emissions_df[state_col] = emissions_df[state_col].astype(int)
# emissions_df = emissions_df.set_index(state_col)
# emissions_df["new_state"] = emissions_df.index.map(state_map)
# emissions_df = emissions_df.sort_values("new_state")

# emissions_df=emissions_df[col_order]
# em_mat = emissions_df[col_order]


In [ ]:
import pandas as pd

# load
emissions_df = pd.read_csv(emissions_in, sep="\t")

state_col = "State (Emission order)"

# ensure numeric
emissions_df[state_col] = emissions_df[state_col].astype(int)

# set old state number as index
emissions_df = emissions_df.set_index(state_col)

# map old state number → new state number
emissions_df["new_state"] = emissions_df.index.map(state_map)

# sort by new state
emissions_df = emissions_df.sort_values("new_state")

# ---- IMPORTANT PART ----
# replace index with new state numbers
emissions_df.index = emissions_df["new_state"]

# drop helper column
emissions_df = emissions_df.drop(columns=["new_state"])

# optional: rename index to ChromHMM style labels
emissions_df.index = [f"A{i}" for i in emissions_df.index]

# reorder columns
emissions_df = emissions_df[col_order]

# ---- save ----
emissions_df.to_csv(emissions_out, sep="\t")

print("emissions.txt reordered and saved.")

# matrix for plotting
em_mat = emissions_df.copy()
em_mat


In [ ]:
import pandas as pd

# load
overlap_df = pd.read_csv(overlap_in, sep="\t", index_col= "State (Emission order)")
state_col = "State (Emission order)"


# overlap_df = emissions_df.set_index(state_col)

# map old state number → new state number
overlap_df["new_state"] = overlap_df.index.map(state_map_str)

# sort by new state
overlap_df = overlap_df.sort_values("new_state")
overlap_df.index = overlap_df["new_state"]

# optional: rename index to ChromHMM style labels
overlap_df.index = [f"A{i}" for i in overlap_df.index]
overlap_df = overlap_df.rename(index={"A24": "Base"})

# drop helper column
overlap_df = overlap_df.drop(columns=["new_state"])

overlap_df.to_csv(overlap_out, sep="\t")

In [ ]:
overlap_df

In [ ]:
# Keep only numeric columns (the marks)
em_mat = em_mat.select_dtypes(include=[np.number]).copy()

# ChromHMM white to blue colormap
chromhmm_cmap = LinearSegmentedColormap.from_list(
    "chromhmm_white_blue",
    ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
)

# plt.figure(figsize=(1.1 * em_mat.shape[1] + 4, 0.35 * em_mat.shape[0] + 4))
# plt.figure(figsize=(20,5))

    
plt.figure(
    figsize=(0.25 * em_mat.shape[1] + 4,
             0.2 * em_mat.shape[0] + 4)
)

ax = sns.heatmap(
    em_mat,
    cmap=chromhmm_cmap,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.3,
    cbar=False,
    linecolor="lightgray",
    # cbar_kws={"label": "Emission probability"}
)

ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xlabel("Mark")
ax.set_ylabel("State")
ax.set_title("ChromHMM Emissions")

plt.tight_layout()
# plt.savefig("chromhmm_emissions_heatmap.pdf")
# plt.savefig("chromhmm_emissions_heatmap.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

def plot_emissions_with_rowgroups(em_mat, row_groups):

    # Keep only numeric columns
    em_mat = em_mat.select_dtypes(include=[np.number]).copy()

    # Reorder rows
    ordered_rows = []
    group_sizes = []

    for gname, rows in row_groups.items():
        ordered_rows.extend(rows)
        group_sizes.append((gname, len(rows)))

    em_mat = em_mat.loc[ordered_rows]

    # Compute group boundaries and centers
    group_boundaries = []
    group_centers = []

    running = 0
    for gname, sz in group_sizes:
        start = running
        end = running + sz
        group_boundaries.append(end)
        group_centers.append((gname, (start + end) / 2.0))
        running = end

    # ChromHMM colormap
    chromhmm_cmap = LinearSegmentedColormap.from_list(
        "chromhmm_white_blue",
        ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
    )

    fig =plt.figure(
        figsize=(0.4 * em_mat.shape[1] + 4,
                 0.2 * em_mat.shape[0] + 4)
    )

    ax = sns.heatmap(
        em_mat,
        cmap=chromhmm_cmap,
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar=False,
        linecolor="lightgray"
    )
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    # ax.set_xlabel("Mark")
    # ax.set_ylabel("State")
    ax.set_title("ChromHMM Emissions")

    # # Horizontal group separators
    for b in group_boundaries[:-1]:
        ax.axhline(b, color="black", linewidth=1.5)

    # for b in group_boundaries[:-1]:
    #     ax.axhline(b - 0.5, color="white", linewidth=6)  # create gap
    #     ax.axhline(b - 0.5, color="black", linewidth=1.5)  # draw line on top


    # Add bold group labels
    for gname, ycenter in group_centers:
        ax.text(
            x=-0.05,
            y=ycenter,
            s=gname,
            transform=ax.get_yaxis_transform(),
            ha="right",
            va="center",
            fontsize=16,
            fontweight="bold"
        )

    # --------- Add OUTER BOUNDARY BOX ----------
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_color("black")

    plt.subplots_adjust(left=0.22)
    plt.tight_layout()
    return fig
    # plt.show()


In [ ]:
row_groups = {
    "Cons open": ['A1','A2'],
    "Mostly open": ['A3','A4','A5','A6'],
    "MEF open": [ 'A7'],
   "MEF CEBP open": [  'A8','A9'],
    "NSC open": [  'A10'],
    "NSC CEBP open": [  'A11'],
    "NSC Cons open": [  'A12'],
    "ESC open": [  'A13'],
    "ESC CEBP open": [  'A14','A15'],
    "ESC Cons open": [  'A16'],
    "Bcell open": [  'A17'],
    "Bcell CEBP open": [  'A18'],
    "Bcell Cons open": [  'A19'],
    "MEF NSC ESC CEBP open": ['A20'],
    "Const close": ['A21','A22','A23']
}


fig = plot_emissions_with_rowgroups(em_mat, row_groups)
fig.savefig(emissions_out_pdf, bbox_inches="tight")

In [ ]:
emissions_out

In [ ]:
seg = pd.read_csv(
    seg_in,
    sep="\t",
    header=None,
    usecols=[0,1,2,3],
    names=["chr","start","end","state"]
)

In [ ]:
seg.head()

In [ ]:
import re

def extract_state_number(x):
    x = str(x)
    if x.startswith("E"):
        return int(x[1:])
    return int(x)

seg["state_num"] = seg["state"].apply(extract_state_number)
seg["new_state_num"] = seg["state_num"].map(state_map)
seg["state"] = "A" + seg["new_state_num"].astype(int).astype(str)


In [ ]:
seg.head()

In [ ]:
seg[["chr","start","end","state"]].to_csv(
    seg_out,
    sep="\t",
    header=False,
    index=False
)

print("Segmentation BED reordered and saved.")


In [ ]:
seg[seg['state']=='A13'].head()

In [ ]:
emissions_with_percentage_df=overlap_df[['Genome %']].merge(emissions_df, left_index=True, right_index=True, how="outer")

emissions_with_percentage_df.drop('Base', inplace=True)
emissions_with_percentage_df=emissions_with_percentage_df.reindex(emissions_df.index)

emissions_with_percentage_df.to_csv(emissions_out_with_Percent, sep="\t")
# emissions_out_with_Percent

In [ ]:
emissions_with_percentage_df

In [ ]:
# Keep only numeric columns (the marks)
em_mat = emissions_with_percentage_df.select_dtypes(include=[np.number]).copy()

# ChromHMM white to blue colormap
chromhmm_cmap = LinearSegmentedColormap.from_list(
    "chromhmm_white_blue",
    ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
)

# plt.figure(figsize=(1.1 * em_mat.shape[1] + 4, 0.35 * em_mat.shape[0] + 4))
# plt.figure(figsize=(20,5))

    
plt.figure(
    figsize=(0.25 * em_mat.shape[1] + 4,
             0.2 * em_mat.shape[0] + 4)
)

ax = sns.heatmap(
    em_mat,
    cmap=chromhmm_cmap,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.3,
    cbar=False,
    linecolor="lightgray",
    # cbar_kws={"label": "Emission probability"}
)

ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xlabel("Mark")
ax.set_ylabel("State")
ax.set_title("ChromHMM Emissions")

plt.tight_layout()
# plt.savefig("chromhmm_emissions_heatmap.pdf")
# plt.savefig("chromhmm_emissions_heatmap.png", dpi=300)
plt.show()
